# Population

Two decades of gridded population estimates and population density for 339 municipalities: the denominator for per-capita indicators and population-weighted averages.

This notebook gathers every code box of the indicator's page, [https://quarcs-lab.github.io/gdo4bolivia/indicators/municipalities339-pop/](https://quarcs-lab.github.io/gdo4bolivia/indicators/municipalities339-pop/) (GeoDevelopment Observatory of Bolivia), so you can work with the data in one place. Run **Setup** first; then any section runs on its own (Runtime › Run all runs everything).

## Ask Gemini, Colab's AI

Colab has an AI assistant, Gemini. Open it with the Gemini button (✦) in Colab's toolbar, or click **Generate** in an empty code cell, and paste one of these questions. It writes and explains code that builds on the data loaded below. Run Setup first, and check its answers against the variables' descriptions on the page.

- "Plot the 10 fastest-growing municipalities by population, 2001-2020."
- "Map the population growth rate between 2001 and 2020 per municipality and summarise it by department."
- "What share of Bolivia's population lives in the 10 largest municipalities, and how did it change from 2001 to 2020?"
- "Map log population density (ln_pop_density) in 2020 and check whether it clusters spatially with geo.moran."
- "Did population concentrate or spread out? Run geo.sigma_convergence("pop", "pop") and explain the result."


## Variables

The columns of `pop`, one row per unit and year. Each one's full description (how it was built, its source and reference years) is under Variables on the page, and in `geo.dictionary("pop")`.

| Variable | Label | Unit |
| --- | --- | --- |
| `geo_id` | Municipality ID |  |
| `year` | Year |  |
| `asdf_id` | DS4Bolivia polygon ID |  |
| `pop` | Population | people |
| `pop_density` | Population density (people per km²) | people per km² |
| `ln_pop_density` | Log population density | ln(people per km²) |


## Setup

Installs the packages every section uses and imports them once.

In [ ]:
!pip install -q geo4bolivia geolibre
import geo4bolivia as geo
import plotly.express as px
import pandas as pd
import geopandas as gpd
from geolibre import Map

## Load the data with the package

In [ ]:
pop = geo.load("pop")

# Totals by department, 2001-2020 (population sums up the hierarchy)
dep = geo.aggregate(pop, to="department", dataset="pop")
names = geo.units("department").set_index("geo_id")["name"]
dep["department"] = dep["geo_id"].map(names)
px.line(dep, x="year", y="pop", color="department",
labels={"pop": "Population"}, title="Population by department, 2001-2020")

## Load the data without the package (plain URLs)

In [ ]:
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
pop = pd.read_csv(BASE + "data/pop/map339/pop.csv")
units = pd.read_csv(BASE + "geo/units.csv")   # every scale and map
units = units[units["scale"].isin(["map339", "department"])]
pop = pop.merge(units[["geo_id", "department_id"]], on="geo_id")

# Totals by department, 2001-2020
dep = pop.groupby(["year", "department_id"], as_index=False)["pop"].sum()
dep["department"] = dep["department_id"].map(units.set_index("geo_id")["name"])
px.line(dep, x="year", y="pop", color="department",
labels={"pop": "Population"}, title="Population by department, 2001-2020")

## Load the data from the wide GeoPackage (one row per municipality)

In [ ]:
# No geo4bolivia needed: one row per municipality, a column per variable
# and year (<variable>_<year>), with boundaries
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
gdf = gpd.read_file(BASE + "data/pop/map339/pop_wide.gpkg",
                    layer="pop")   # or pop_wide.geojson (lighter)
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="pop_density_2020", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"pop_density_2020": "Population density (people per km²)"},
          title="Population density (people per km²), 2020")

## Load the data in GeoLibre (the wide GeoJSON)

In [ ]:
# GeoLibre in the notebook: the wide GeoJSON, one feature per municipality
# and a column per variable and year (<variable>_<year>); restyle any of
# them in the Style panel
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
m = Map(center=[-64.5, -16.5], zoom=4.5, height="600px")
m.add_choropleth(BASE + "data/pop/map339/pop_wide.geojson",
                 column="ln_pop_density_2020", name="Log population density, 2020",
                 strokeColor="#ffffff", strokeWidth=0.3)
m

## License and how to cite

The data are published under [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/); their sources keep their own licenses:

- WorldPop (University of Southampton, University of Louisville, Université de Namur) and CIESIN, Columbia University: [Creative Commons Attribution 4.0 International License](https://hub.worldpop.org/doi/10.5258/SOTON/WP00647)
- AidData, William & Mary: [Open Data Commons Attribution License 1.0](https://opendatacommons.org/licenses/by/1-0/)
- geoBoundaries, William & Mary geoLab: [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/)
- Carlos Mendez, Erick Gonzales, Pedro Leoni, Lykke Andersen and Hernán Peralta: [MIT License](https://github.com/quarcs-lab/ds4bolivia/blob/master/LICENSE)

If you use these data, please cite, in this order, the original source, the observatory, which processes and maintains them, and the other sources:

1. WorldPop & CIESIN (2018). Global High Resolution Population Denominators Project (1 km aggregated, 2000–2020). University of Southampton and Columbia University. Funded by the Bill and Melinda Gates Foundation (OPP1134076).
2. Mendez, C. (2026). GeoDevelopment Observatory of Bolivia. https://quarcs-lab.github.io/gdo4bolivia/
3. Goodman, S., BenYishay, A., Lv, Z., & Runfola, D. (2019). GeoQuery: Integrating HPC systems and public web-based geospatial data tools. Computers & Geosciences, 122, 103–112.
4. Runfola, D., Anderson, A., Baier, H., et al. (2020). geoBoundaries: A global database of political administrative boundaries. PLoS ONE, 15(4), e0231866.
5. Mendez, C., Gonzales, E., Leoni, P., Andersen, L., & Peralta, H. (2026). DS4Bolivia: A Data Science Repository to Study GeoSpatial Development in Bolivia [Data set]. GitHub.
